# 05 — Dashboard Pipeline & Tableau Export

**Goal:** End-to-end pipeline that turns raw Telco data into the CSVs that power the Tableau dashboard.

**Flow:**
1. Load the cleaned customer data.
2. Generate **unbiased** churn probabilities for every customer using 5-fold cross-val prediction (so the dashboard reflects what the model would say for never-seen rows, not optimistically scoring its own training set).
3. Apply the optimal threshold from notebook 04 to flag who to target.
4. Load `customers` + `predictions` into a SQLite database.
5. Run the SQL feature-engineering and cohort views (`sql/01_*.sql`, `sql/02_*.sql`).
6. Read views back and export Tableau-ready CSVs to `data/dashboard/`.

**Why SQL here?** The README calls for feature engineering "encoded in SQL to mirror real analyst workflows." Anyone with read access to the warehouse can answer cohort questions without touching Python.

In [ ]:
import sqlite3
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from src.config import PROJECT_ROOT, RANDOM_SEED, SQL_DIR
from src.cost_analysis import CostAssumptions, optimal_threshold, sweep_thresholds
from src.data_loader import load_clean
from src.features import make_features
from src.model import train_xgboost

DB_PATH = PROJECT_ROOT / "data" / "churn.db"
DASHBOARD_DIR = PROJECT_ROOT / "data" / "dashboard"
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)
print(f"DB:        {DB_PATH}")
print(f"Exports:   {DASHBOARD_DIR}")

## 1. Load and featurize the full dataset

In [ ]:
df = load_clean()
X, y = make_features(df)
print(f"Customers: {len(df)} | Features: {X.shape[1]} | Churn rate: {y.mean():.1%}")

## 2. Out-of-fold predictions (every customer scored by a model that didn't see them)

5-fold stratified CV. Slower than a single fit but the only honest way to score the training data — avoids the optimistic-bias trap where the dashboard shows P(churn) ≈ 0 for everyone the model overfit on.

In [ ]:
# We need a fresh estimator instance per call; train_xgboost fits, but
# cross_val_predict needs an unfitted estimator with sklearn's interface.
from xgboost import XGBClassifier

scale_pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
estimator = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc", random_state=RANDOM_SEED, n_jobs=-1,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
y_proba = cross_val_predict(estimator, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
print(f"Out-of-fold predictions: shape={y_proba.shape}, mean={y_proba.mean():.3f}")

## 3. Pick the operating threshold via cost-sensitive sweep

Same logic as notebook 04, applied to the full out-of-fold predictions so the threshold reflects the population the dashboard represents.

In [ ]:
assumptions = CostAssumptions(intervention_cost=50.0, success_rate=0.30, clv_horizon_months=12)
sweep = sweep_thresholds(y, y_proba, df["MonthlyCharges"], assumptions)
best = optimal_threshold(sweep)
tau_star = best["threshold"]
print(f"Optimal threshold (population): {tau_star:.2f}")
print(f"Net value at tau*: ${best['net_value']:,.0f} | Targeting {int(best['n_targeted'])} customers")

In [ ]:
predictions = pd.DataFrame({
    "customerID": df["customerID"].values,
    "churn_probability": y_proba,
    "is_targeted": (y_proba >= tau_star).astype(int),
})
predictions.head()

## 4. Load `customers` + `predictions` into SQLite

We rebuild the DB from scratch on each run so the export is reproducible and never carries stale rows from a previous experiment.

In [ ]:
if DB_PATH.exists():
    DB_PATH.unlink()

with sqlite3.connect(DB_PATH) as conn:
    df.to_sql("customers", conn, index=False)
    predictions.to_sql("predictions", conn, index=False)
    n_customers = conn.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
    n_predictions = conn.execute("SELECT COUNT(*) FROM predictions").fetchone()[0]

print(f"Loaded customers={n_customers}, predictions={n_predictions}")

## 5. Execute the SQL files

`executescript` runs all statements in the file. We re-create views every time (`DROP VIEW IF EXISTS` is built into the SQL files) so iteration on the SQL doesn't require deleting the DB.

In [ ]:
sql_files = sorted(SQL_DIR.glob("*.sql"))
with sqlite3.connect(DB_PATH) as conn:
    for sql_file in sql_files:
        print(f"Running {sql_file.name}")
        conn.executescript(sql_file.read_text())

    views = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='view' ORDER BY name", conn
    )
print("\nViews now defined:")
print(views.to_string(index=False))

## 6. Read views back and inspect

Quick eyeball check that the SQL produced sensible output before we export.

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    customer_features = pd.read_sql("SELECT * FROM customer_features", conn)
    at_risk_cohorts = pd.read_sql("SELECT * FROM at_risk_cohorts", conn)
    cohort_summary = pd.read_sql("SELECT * FROM cohort_summary", conn)
    churn_by_tc = pd.read_sql("SELECT * FROM churn_by_tenure_contract", conn)

print(f"customer_features:        {customer_features.shape}")
print(f"at_risk_cohorts:          {at_risk_cohorts.shape}")
print(f"cohort_summary:           {cohort_summary.shape}")
print(f"churn_by_tenure_contract: {churn_by_tc.shape}")

In [ ]:
cohort_summary.round(3)

In [ ]:
churn_by_tc.round(3)

## 7. Headline KPIs

These are the numbers that go on the dashboard's top-row cards. Same numbers that should populate the README's Headline Results section.

In [ ]:
total_mrr = df["MonthlyCharges"].sum()
expected_at_risk = at_risk_cohorts["expected_revenue_at_risk"].sum()
n_targeted = int(predictions["is_targeted"].sum())

print(f"Customer base:                {len(df):,}")
print(f"Total monthly recurring rev:  ${total_mrr:>12,.0f}")
print(f"Expected 12-mo revenue at risk: ${expected_at_risk:>12,.0f}")
print(f"Customers to target (tau*={tau_star:.2f}): {n_targeted:,}")
print(f"Net value of intervention:    ${best['net_value']:>12,.0f}")

## 8. Export Tableau-ready CSVs

One CSV per view — Tableau handles CSVs natively and can refresh from the same files when the pipeline reruns.

In [ ]:
exports = {
    "customer_features.csv": customer_features,
    "at_risk_cohorts.csv": at_risk_cohorts,
    "cohort_summary.csv": cohort_summary,
    "churn_by_tenure_contract.csv": churn_by_tc,
}

for filename, frame in exports.items():
    out = DASHBOARD_DIR / filename
    frame.to_csv(out, index=False)
    print(f"  wrote {out.relative_to(PROJECT_ROOT)}  ({len(frame):,} rows)")

## 9. Handoff to Tableau

**Connect Tableau to:** `data/dashboard/at_risk_cohorts.csv` as the primary source. Join it to the others on `priority_bucket` / `tenure_bucket` as needed.

**Recommended dashboard views:**
- KPI cards (top): customer count, total MRR, expected revenue at risk, customers targeted at $\tau^*$.
- Cohort treemap: priority_bucket × MRR (size) × avg churn_probability (color).
- Heatmap: churn rate by `tenure_bucket × Contract` (the EDA's clearest segmentation).
- Table: top-N highest `expected_revenue_at_risk` customers with action recommendation.

**To refresh:** rerun this notebook → CSVs overwrite → Tableau "Refresh Data Source".